<a href="https://colab.research.google.com/github/peremartra/CH13/blob/main/CH12_NB01_vllm_attention_free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Serving a model with attention-free blocks in vLLM

`oopere/llama-3.2-3b-attn-drop-6` is Llama-3.2-3B with the attention sub-block removed from 6 of its 28 decoder blocks (18, 20, 21, 22, 23 and 24). Those blocks keep their MLP, so the model still has 28 blocks but only 22 of them have attention, and only those 22 need a KV cache.

The KV cache formula predicts the saving per token, in float16:

$$2 \times 8 \text{ KV heads} \times 128 \text{ head dim} \times 2 \text{ bytes} \times L_{attn}$$

which gives 114,688 bytes/token for 28 attention blocks and 90,112 for 22. Whether you get that saving depends on the inference engine, not on the checkpoint. This notebook checks it in vLLM:

1. An unmodified Llama-3.2-3B as the control.
2. The pruned model loaded the standard way. **It fails**, and we look at why.
3. The pruned model loaded with a small custom model class registered as a vLLM plugin. **It works**, and we measure KV cache and latency against the control.

The pruned model has not been through knowledge distillation, so it has lost some capability compared with the original. This notebook doesn't evaluate its quality. It only checks that vLLM serves it correctly: same output as the Transformers implementation on a simple prompt.

Tested on a Colab NVIDIA L4 with vLLM 0.30.0, in float16. Each vLLM load runs in its own process, so a failed load cannot leave the notebook's GPU in a bad state, and every load writes a full log to disk.

## 1. Setup

In [64]:
%pip -q install vllm==0.30.0 huggingface_hub

Installing vLLM can replace the torch version that Colab ships. This notebook never imports torch or vLLM in the kernel itself, only in subprocesses, so no restart is needed. The next cell prints the versions a fresh process sees.

In [65]:
import json, re, subprocess, sys
from pathlib import Path

WORK_DIR = Path("ch12_vllm")
(WORK_DIR / "logs").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "attn_drop_plugin" / "attn_drop_vllm").mkdir(parents=True, exist_ok=True)

VERSIONS = r'''
import json, torch, transformers, vllm
print(json.dumps({"vllm": vllm.__version__, "torch": torch.__version__,
                  "transformers": transformers.__version__, "cuda": torch.version.cuda,
                  "gpu": torch.cuda.get_device_name(0)}, indent=2))
'''
print(subprocess.run([sys.executable, "-c", VERSIONS], capture_output=True, text=True).stdout)

{
  "vllm": "0.30.0",
  "torch": "2.13.0+cu130",
  "transformers": "5.17.0",
  "cuda": "13.0",
  "gpu": "NVIDIA L4"
}



In [66]:
TARGET_ID = "oopere/llama-3.2-3b-attn-drop-6"
CONTROL_ID = "unsloth/Llama-3.2-3B"   # ungated copy of meta-llama/Llama-3.2-3B

# One configuration for every load, so the comparison is fair.
# float16 rather than bfloat16, so the same notebook also runs on a T4.
LLM_KWARGS = dict(dtype="half", max_model_len=2048, gpu_memory_utilization=0.85,
                  enforce_eager=True, trust_remote_code=True)

KV_HEADS, HEAD_DIM, BYTES_FP16 = 8, 128, 2
PER_BLOCK = 2 * KV_HEADS * HEAD_DIM * BYTES_FP16       # K and V for one attention block
FORMULA = {"28 attention blocks": PER_BLOCK * 28, "22 attention blocks": PER_BLOCK * 22}
print(FORMULA)

{'28 attention blocks': 114688, '22 attention blocks': 90112}


## 2. What the checkpoint contains

In [67]:
from huggingface_hub import hf_hub_download

target_config = json.load(open(hf_hub_download(TARGET_ID, "config.json")))
DROPPED = target_config["dropped_attn_layers"]
print("architectures      :", target_config["architectures"])
print("num_hidden_layers  :", target_config["num_hidden_layers"])
print("dropped_attn_layers:", DROPPED)

# Compare the tensors stored for an intact block and for an attention-free one
weight_map = json.load(open(hf_hub_download(TARGET_ID, "model.safetensors.index.json")))["weight_map"]
for layer in (17, DROPPED[0]):
    names = sorted(k.split(f"layers.{layer}.")[1] for k in weight_map if f".layers.{layer}." in k)
    print(f"\nblock {layer}:", names)

architectures      : ['LlamaForCausalLM']
num_hidden_layers  : 28
dropped_attn_layers: [18, 20, 21, 22, 23, 24]

block 17: ['input_layernorm.weight', 'mlp.down_proj.weight', 'mlp.gate_proj.weight', 'mlp.up_proj.weight', 'post_attention_layernorm.weight', 'self_attn.k_proj.weight', 'self_attn.o_proj.weight', 'self_attn.q_proj.weight', 'self_attn.v_proj.weight']

block 18: ['mlp.down_proj.weight', 'mlp.gate_proj.weight', 'mlp.up_proj.weight', 'post_attention_layernorm.weight']


The attention-free blocks have no `q/k/v/o_proj` and no `input_layernorm`, the norm that feeds attention. Only `post_attention_layernorm` and the MLP remain. The config still declares the standard `LlamaForCausalLM` architecture. The repo also ships a custom Transformers class through `auto_map`, which Transformers uses with `trust_remote_code=True`.

## 3. The vLLM probe

One script, run in a fresh process for every model. It loads the model with `LLM_KWARGS` and, if the load succeeds, it runs two tests:

- **Free generation:** greedy, 30 tokens. We compare it with a Transformers reference.
- **Batch-1 latency:** median of 10 runs after a warm-up.

vLLM writes the KV cache it allocated to its log. We read it from there.

In [68]:
%%writefile ch12_vllm/vllm_probe.py
import json, statistics, sys, time, traceback

model_id, out_path, kwargs_json = sys.argv[1], sys.argv[2], sys.argv[3]
result = {"model": model_id, "loaded": False}

PROMPT = "The three most visited cities in Europe are"
N_TOKENS = 30

try:
    from vllm import LLM, SamplingParams

    llm = LLM(model=model_id, **json.loads(kwargs_json))
    result["loaded"] = True

    # ignore_eos keeps every run at exactly N_TOKENS, so runs are comparable
    free = SamplingParams(temperature=0.0, max_tokens=N_TOKENS, ignore_eos=True)
    result["free_text"] = llm.generate([PROMPT], free)[0].outputs[0].text   # also the warm-up

    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        llm.generate([PROMPT], free, use_tqdm=False)
        times.append((time.perf_counter() - t0) * 1000)
    result["ms_per_token"] = round(statistics.median(times) / N_TOKENS, 2)
except Exception:
    traceback.print_exc()

json.dump(result, open(out_path, "w"), indent=2)

Overwriting ch12_vllm/vllm_probe.py


In [69]:
def run_vllm(model_id, name, extra_kwargs=None):
    # Runs the probe in its own process; the full stdout/stderr goes to a log file
    out_json, log_path = WORK_DIR / f"{name}.json", WORK_DIR / "logs" / f"{name}.log"
    kwargs = {**LLM_KWARGS, **(extra_kwargs or {})}
    with open(log_path, "w") as log:
        subprocess.run([sys.executable, str(WORK_DIR / "vllm_probe.py"), model_id, str(out_json),
                        json.dumps(kwargs)], stdout=log, stderr=subprocess.STDOUT)
    result = json.loads(out_json.read_text()) if out_json.exists() else {"loaded": False}
    log_text = log_path.read_text(errors="replace")

    mem = re.search(r"Available KV cache memory:\s*([\d.]+)\s*GiB", log_text)
    tok = re.search(r"GPU KV cache size:\s*([\d,]+)\s*tokens", log_text)
    if mem and tok:
        result["kv_gib"] = float(mem.group(1))
        result["kv_tokens"] = int(tok.group(1).replace(",", ""))
        result["kv_bytes_per_token"] = round(result["kv_gib"] * 2**30 / result["kv_tokens"])

    if not result["loaded"]:
        # vLLM's last line ("Engine core initialization failed") is a wrapper;
        # the first exception in the log is the real cause
        result["first_error"] = next(
            (l for l in log_text.splitlines()
             if re.search(r"\b\w*(Error|Exception):", l) and "Engine core initialization failed" not in l),
            None)
    print(f"{name}: loaded={result['loaded']}  (log: {log_path})")
    return result

## 4. Control: an unmodified Llama-3.2-3B

In [70]:
control = run_vllm(CONTROL_ID, "control")
print(f"KV cache: {control['kv_gib']} GiB, {control['kv_tokens']:,} tokens, "
      f"{control['kv_bytes_per_token']:,} bytes/token  (formula: {FORMULA['28 attention blocks']:,})")

control: loaded=True  (log: ch12_vllm/logs/control.log)
KV cache: 11.97 GiB, 112,016 tokens, 114,740 bytes/token  (formula: 114,688)


The measured bytes per token match the formula for 28 attention blocks. The small gap comes from vLLM rounding the GiB figure to two decimals. This also confirms that the method used to read the KV cache from the log works.

## 5. The pruned model, the standard way

In [71]:
standard = run_vllm(TARGET_ID, "attn_drop_standard")
print(standard.get("first_error"))

attn_drop_standard: loaded=False  (log: ch12_vllm/logs/attn_drop_standard.log)
(EngineCore pid=36419) ERROR 10-04 17:05:10 [core.py:1366] ValueError: Following weights were not initialized from checkpoint: {'model.layers.20.input_layernorm.weight', 'model.layers.23.input_layernorm.weight', 'model.layers.22.input_layernorm.weight', 'model.layers.18.input_layernorm.weight', 'model.layers.24.input_layernorm.weight', 'model.layers.21.input_layernorm.weight'}


The load fails after reading the weights:

```
ValueError: Following weights were not initialized from checkpoint:
{'model.layers.18.input_layernorm.weight', 'model.layers.20.input_layernorm.weight', ...}
```

vLLM does not run the repo's custom Transformers class. It reads `architectures: ["LlamaForCausalLM"]` from the config and builds its own optimized Llama, which assumes that all 28 blocks are identical: every block has attention and the norm that feeds it. The checkpoint doesn't have those weights for the 6 pruned blocks, so the load stops. `trust_remote_code=True` doesn't change this. The checkpoint is fine; the engine's model definition is what is missing.

## 6. The pruned model with a custom vLLM model class

vLLM lets you register your own architectures through a plugin. A plugin is a small Python package with a `vllm.general_plugins` entry point, which vLLM loads in every process, including the engine core. The class does not need to rewrite Llama. vLLM's `LlamaForCausalLM` accepts a `layer_type` argument, so we only need a decoder layer that, for the blocks listed in `dropped_attn_layers`:

- builds only `post_attention_layernorm` and the MLP, which matches what the checkpoint stores;
- computes `x + mlp(norm(x))`, the same as the repo's Transformers class;
- registers no `Attention` layer. vLLM allocates KV cache per attention layer, so these blocks get none.

In [72]:
%%writefile ch12_vllm/attn_drop_plugin/attn_drop_vllm/model.py
from torch import nn

from vllm.model_executor.layers.layernorm import RMSNorm
from vllm.model_executor.models.llama import (
    LlamaAttention, LlamaDecoderLayer, LlamaForCausalLM, LlamaMLP)
from vllm.model_executor.models.utils import extract_layer_index


class AttnFreeAwareDecoderLayer(LlamaDecoderLayer):
    def __init__(self, vllm_config, prefix="", config=None, attn_layer_type=LlamaAttention):
        config = config or vllm_config.model_config.hf_config
        dropped = set(getattr(config, "dropped_attn_layers", None) or [])
        if extract_layer_index(prefix) not in dropped:
            # A normal block: vLLM's own implementation, unchanged
            super().__init__(vllm_config=vllm_config, prefix=prefix, config=config,
                             attn_layer_type=attn_layer_type)
            self.has_attention = True
            return

        # An attention-free block: only the MLP and its norm, no Attention layer
        nn.Module.__init__(self)
        self.has_attention = False
        self.hidden_size = config.hidden_size
        self.mlp = LlamaMLP(
            hidden_size=config.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=config.hidden_act,
            quant_config=self.get_quant_config(vllm_config),
            bias=getattr(config, "mlp_bias", False),
            prefix=f"{prefix}.mlp",
        )
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, positions, hidden_states, residual):
        if self.has_attention:
            return super().forward(positions, hidden_states, residual)
        # vLLM carries the residual stream as (pending output, residual). The fused
        # add + norm folds the previous block's output in, so this is x + mlp(norm(x))
        if residual is None:
            residual = hidden_states
            hidden_states = self.post_attention_layernorm(hidden_states)
        else:
            hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class AttnDropLlamaForCausalLM(LlamaForCausalLM):
    def __init__(self, *, vllm_config, prefix=""):
        if not getattr(vllm_config.model_config.hf_config, "dropped_attn_layers", None):
            raise ValueError("AttnDropLlamaForCausalLM needs config.dropped_attn_layers")
        super().__init__(vllm_config=vllm_config, prefix=prefix,
                         layer_type=AttnFreeAwareDecoderLayer)

Overwriting ch12_vllm/attn_drop_plugin/attn_drop_vllm/model.py


In [73]:
%%writefile ch12_vllm/attn_drop_plugin/attn_drop_vllm/__init__.py
def register():
    from vllm import ModelRegistry
    # A "module:Class" string keeps the import lazy, so loading the plugin doesn't touch CUDA
    if "AttnDropLlamaForCausalLM" not in ModelRegistry.get_supported_archs():
        ModelRegistry.register_model("AttnDropLlamaForCausalLM",
                                     "attn_drop_vllm.model:AttnDropLlamaForCausalLM")

Overwriting ch12_vllm/attn_drop_plugin/attn_drop_vllm/__init__.py


In [74]:
%%writefile ch12_vllm/attn_drop_plugin/pyproject.toml
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "attn-drop-vllm"
version = "0.1.0"

[project.entry-points."vllm.general_plugins"]
attn_drop = "attn_drop_vllm:register"

[tool.setuptools]
packages = ["attn_drop_vllm"]

Overwriting ch12_vllm/attn_drop_plugin/pyproject.toml


In [75]:
%pip -q install --no-deps --force-reinstall ./ch12_vllm/attn_drop_plugin

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


The checkpoint stays as it is. `hf_overrides` changes the config only in memory: it points vLLM to the new architecture name and passes the list of attention-free blocks.

In [76]:
custom = run_vllm(TARGET_ID, "attn_drop_custom", extra_kwargs={
    "hf_overrides": {"architectures": ["AttnDropLlamaForCausalLM"], "dropped_attn_layers": DROPPED}})

attn_drop_custom: loaded=True  (log: ch12_vllm/logs/attn_drop_custom.log)


## 7. Does the custom class compute the same model?

This is a functional check, not a quality evaluation. The reference is the repo's own Transformers class, with the same prompt, greedy decoding and float16. If vLLM and Transformers produce the same tokens, the custom class implements the same function as the original model. The reference also runs in a subprocess, so it releases the GPU when it finishes.

Both outputs come from the same pruned model, so they should match even if the text is poor. For contrast, the cell also prints what the unpruned control model wrote for the same prompt in vLLM.

In [ ]:
REFERENCE = r'''
import json, sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id, out_path = sys.argv[1], sys.argv[2]
PROMPT = "The three most visited cities in Europe are"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda",
                                             trust_remote_code=True).eval()
inputs = tok(PROMPT, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=30, min_new_tokens=30, do_sample=False)
json.dump({"free_text": tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)},
          open(out_path, "w"))
'''
(WORK_DIR / "hf_reference.py").write_text(REFERENCE)
subprocess.run([sys.executable, str(WORK_DIR / "hf_reference.py"), TARGET_ID, str(WORK_DIR / "hf_reference.json")],
               stdout=open(WORK_DIR / "logs" / "hf_reference.log", "w"), stderr=subprocess.STDOUT)
reference = json.loads((WORK_DIR / "hf_reference.json").read_text())

print("Transformers:", repr(reference["free_text"]))
print("vLLM custom :", repr(custom["free_text"]))
print("Identical   :", reference["free_text"] == custom["free_text"])

# For contrast, the unpruned model on the same prompt (from the control run)
print("\nUnpruned    :", repr(control["free_text"]))

## 8. Control vs. pruned model in vLLM

In [78]:
rows = [
    ("Memory vLLM reserves for the KV cache (GiB)", "kv_gib", "{}"),
    ("Tokens that fit in that memory", "kv_tokens", "{:,}"),
    ("KV bytes per token", "kv_bytes_per_token", "{:,}"),
    ("Batch-1 latency (ms/token)", "ms_per_token", "{}"),
]
print(f"{'':46}{'Control':>12}{'attn-drop-6':>14}")
for label, key, fmt in rows:
    print(f"{label:46}{fmt.format(control[key]):>12}{fmt.format(custom[key]):>14}")

print(f"\nFormula: 28 blocks = {FORMULA['28 attention blocks']:,} bytes/token, "
      f"22 blocks = {FORMULA['22 attention blocks']:,} bytes/token")
print(f"Tokens that fit, pruned vs control: {custom['kv_tokens'] / control['kv_tokens']:.2f}x")
print(f"Latency, pruned / control         : {custom['ms_per_token'] / control['ms_per_token']:.3f}")

                                                   Control   attn-drop-6
Memory vLLM reserves for the KV cache (GiB)          11.97         12.25
Tokens that fit in that memory                     112,016       145,920
KV bytes per token                                 114,740        90,141
Batch-1 latency (ms/token)                           26.57         25.24

Formula: 28 blocks = 114,688 bytes/token, 22 blocks = 90,112 bytes/token
Tokens that fit, pruned vs control: 1.30x
Latency, pruned / control         : 0.950


## 9. What the numbers say

Results from our run (NVIDIA L4, vLLM 0.30.0, torch 2.13.0, transformers 5.17.0, float16, `enforce_eager=True`):

| | Control (28 attention blocks) | attn-drop-6, custom class (22) |
|---|---|---|
| Memory vLLM reserves for the KV cache | 11.97 GiB | 12.25 GiB |
| Tokens that fit in that memory | 112,016 | 145,920 |
| KV bytes per token | 114,740 (formula: 114,688) | 90,141 (formula: 90,112) |
| Batch-1 latency | 26.57 ms/token | 25.24 ms/token |

- **The first row is not what the model needs.** vLLM takes the share of GPU memory set by `gpu_memory_utilization`, subtracts the weights and the activations, and gives everything left to the KV cache. The pruned model has about 0.28 GiB less of weights, the removed attention projections, so more memory is left for the cache. The cost of the KV cache shows in bytes per token.
- **The KV saving is real, but only with an engine that knows about it.** The standard vLLM Llama refuses the checkpoint. With the custom class, vLLM allocates KV cache only for the 22 attention blocks, exactly as the formula predicts. Each token costs 21% less, and with the extra memory 30% more tokens fit.
- **Batch-1 latency improves only about 5%.** That is roughly the drop in parameters, 3.06B against 3.21B. At batch 1 and short context, decoding time is dominated by reading the weights, not by the KV cache. The KV saving should matter more with long contexts and many concurrent requests, which this notebook doesn't measure.
- **vLLM serves the same model.** Its output matches the Transformers implementation token for token on the test prompt. That doesn't make it a good model, and the text itself ("London, Paris and Paris...") shows it: without knowledge distillation, removing attention costs capability, and measuring that is a separate evaluation.

Not covered here: other vLLM versions, other GPUs, CUDA graphs (`enforce_eager=False`), tensor parallelism, long-context throughput, and model quality.